# 🛒 YOLOv8x — RPC Retail Product Pricing Recognition
**Full end-to-end training pipeline on the Retail Product Checkout (RPC) dataset.**

Runtime: Google Colab + GPU (T4 / A100)  
Dataset: [diyer22/retail-product-checkout-dataset](https://www.kaggle.com/datasets/diyer22/retail-product-checkout-dataset) — 10% subset  
Subset: **10 %** (change `SUBSET_FRAC` below to any value 0–1)  

---
**Run order:** Execute all cells top to bottom. Each section is self-contained.

## 1 · Environment

In [ ]:
!nvidia-smi
!pip install -q ultralytics kagglehub pycocotools opencv-python-headless tqdm

import torch
assert torch.cuda.is_available(), "GPU not detected — enable GPU in Runtime → Change runtime type"
print(f"GPU: {torch.cuda.get_device_name(0)}  |  PyTorch: {torch.__version__}")

In [ ]:
import sys, os

# ── Point to repo root ────────────────────────────────────────────
# If you uploaded the project folder to Colab, set REPO to its path.
# If you cloned from GitHub: !git clone <url> /content/repo
REPO = '/content/YOLOv8x Product Pricing-Recognition'
sys.path.insert(0, REPO)
os.chdir(REPO)

from src.dataset   import RPCDatasetConverter
from src.pricing   import PriceCatalog
from src.train     import train, evaluate
from src.inference import ProductDetector
print('Modules loaded.')

## 2 · Kaggle Credentials

In [ ]:
# Add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets (🔑 icon, left sidebar)
# Get your key at: https://www.kaggle.com/settings → API → Create New Token
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
print('Kaggle credentials set.')

## 3 · Download & Prepare Dataset

In [ ]:
# ── Configuration ─────────────────────────────────────────────────
SUBSET_FRAC  = 0.10          # 0.10 = 10 %;  1.0 = full dataset
OUTPUT_DIR   = '/content/rpc_subset'
DATASET_NAME = 'diyer22/retail-product-checkout-dataset'

In [ ]:
converter = RPCDatasetConverter(output_dir=OUTPUT_DIR)

# Download (~4 GB — takes 2–5 min on first run; cached on reruns)
converter.download_rpc()

In [ ]:
# Convert COCO → YOLO and sample subset
converter.convert(subset_frac=SUBSET_FRAC)

# Write dataset YAML
YAML_PATH = converter.write_yaml()

# Reference the catalogue
catalog = converter.catalog
print(f'\nClasses : {len(catalog)}')
print(f'YAML    : {YAML_PATH}')

In [ ]:
from pathlib import Path

train_imgs = list(Path(OUTPUT_DIR, 'images', 'train').glob('*.jpg'))
val_imgs   = list(Path(OUTPUT_DIR, 'images', 'val').glob('*.jpg'))
print(f'Train images : {len(train_imgs)}')
print(f'Val   images : {len(val_imgs)}')
print(f'YAML preview:')
print(YAML_PATH.read_text()[:400])

## 4 · Preview Annotated Samples

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def show_samples(img_dir, lbl_dir, catalog, n=4):
    imgs = random.sample(sorted(Path(img_dir).glob('*.jpg')), min(n, 20))
    fig, axes = plt.subplots(1, len(imgs), figsize=(5 * len(imgs), 5))
    if len(imgs) == 1: axes = [axes]
    for ax, img_path in zip(axes, imgs):
        arr = np.array(Image.open(img_path))
        H, W = arr.shape[:2]
        ax.imshow(arr)
        lbl = Path(lbl_dir) / (img_path.stem + '.txt')
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                p = line.split()
                if len(p) < 5: continue
                cls, cx, cy, bw, bh = int(p[0]), *map(float, p[1:])
                x1 = (cx - bw/2) * W;  y1 = (cy - bh/2) * H
                ax.add_patch(patches.Rectangle(
                    (x1, y1), bw*W, bh*H,
                    linewidth=1.5, edgecolor='lime', facecolor='none'))
                name  = catalog.get_name(cls)
                price = catalog.get_price(cls)
                ax.text(x1, y1-3, f'{name[:10]} ¥{price:.1f}',
                        color='yellow', fontsize=6,
                        bbox=dict(fc='black', alpha=0.5, pad=1))
        ax.axis('off')
        ax.set_title(img_path.stem[:18], fontsize=7)
    plt.suptitle('Ground-truth annotations (YOLO format)', fontsize=11)
    plt.tight_layout()
    plt.show()

show_samples(
    f'{OUTPUT_DIR}/images/train',
    f'{OUTPUT_DIR}/labels/train',
    catalog,
)

## 5 · Train YOLOv8x

In [ ]:
# All hyper-parameters are in configs/train_config.yaml.
# Override individual values here without editing the YAML.

trained_model = train(
    data        = YAML_PATH,
    config_path = 'configs/train_config.yaml',
    # Uncomment to override specific params:
    # epochs  = 10,   # quick smoke-test
    # batch   = 8,    # if GPU OOM
    # imgsz   = 640,
)

## 6 · Training Curves

In [ ]:
import pandas as pd
from pathlib import Path

# Find the most recent results.csv
csv_files = sorted(Path('runs').rglob('results.csv'))
if not csv_files:
    print('No results.csv found — training may not have completed.')
else:
    df = pd.read_csv(csv_files[-1])
    df.columns = df.columns.str.strip()

    pairs = [
        ('train/box_loss',    'Train Box Loss'),
        ('train/cls_loss',    'Train Cls Loss'),
        ('val/box_loss',      'Val Box Loss'),
        ('val/cls_loss',      'Val Cls Loss'),
        ('metrics/mAP50',     'mAP@0.5'),
        ('metrics/mAP50-95',  'mAP@0.5:0.95'),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, (col, title) in zip(axes.flat, pairs):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], linewidth=1.5)
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
        else:
            ax.set_visible(False)
    plt.suptitle('YOLOv8x — RPC Training Curves', fontsize=13)
    plt.tight_layout()
    plt.savefig('runs/training_curves.png', dpi=120)
    plt.show()

## 7 · Evaluation

In [ ]:
BEST_WEIGHTS = sorted(Path('runs').rglob('best.pt'))[-1]
print(f'Evaluating: {BEST_WEIGHTS}')

metrics = evaluate(
    model_path = BEST_WEIGHTS,
    data       = YAML_PATH,
)

## 8 · Image Inference

In [ ]:
detector = ProductDetector(
    model_path   = BEST_WEIGHTS,
    catalog_path = f'{OUTPUT_DIR}/price_catalog.json',
    conf = 0.30,
    iou  = 0.45,
)

sample = val_imgs[0]
annotated, receipt = detector.infer_image(sample)

plt.figure(figsize=(13, 8))
plt.imshow(annotated)
plt.axis('off')
plt.title(f"{sum(receipt['counts'].values())} products detected  |  Total ¥{receipt['total']:.2f}",
          fontsize=12)
plt.tight_layout()
plt.show()

print('\nItemised receipt:')
for name, cnt, sub in receipt['lines']:
    print(f'  {name:32s} x{cnt:2d}   ¥{sub:.2f}')
print(f'  {"TOTAL":32s}       ¥{receipt["total"]:.2f}')

## 9 · Video Inference
Upload your video via **Files → Upload** then set `VIDEO_INPUT`.

In [ ]:
VIDEO_INPUT  = '/content/retail_checkout.mp4'   # ← change to your file
VIDEO_OUTPUT = '/content/retail_output.mp4'

if not Path(VIDEO_INPUT).exists():
    print('Upload a .mp4 via the Files panel on the left, then re-run this cell.')
else:
    totals = detector.process_video(
        video_path  = VIDEO_INPUT,
        output_path = VIDEO_OUTPUT,
        skip_frames = 1,          # set 2–3 for faster processing
    )
    print(f"\nSession total: ¥{totals['total']:.2f}")

    from google.colab import files
    files.download(VIDEO_OUTPUT)

## 10 · Export Weights

In [ ]:
import zipfile
from ultralytics import YOLO
from google.colab import files

# ONNX export
model = YOLO(str(BEST_WEIGHTS))
model.export(format='onnx', imgsz=640, dynamic=True, opset=17)
print('ONNX exported.')

# Zip weights + catalogue
zip_path = '/content/yolov8x_rpc_weights.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    zf.write(str(BEST_WEIGHTS), 'best.pt')
    zf.write(f'{OUTPUT_DIR}/price_catalog.json', 'price_catalog.json')
    zf.write(str(YAML_PATH), 'dataset.yaml')

files.download(zip_path)
print('Download started.')